In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import pint
import pint_xarray
from imagematerials.util import dataset_to_array
from imagematerials.util import import_from_netcdf, export_to_netcdf

from constants import vehicle_type_mapping, target_regions, region_mapping, target_variables_slow, target_variables_narrow
# Import Preprocesssed data
vehicle_pre = import_from_netcdf("../../../examples/prep_vema2.nc")
# ['battery_materials', 'battery_shares', 'weight_boats', 'material_fractions', 'vehicle_weights', 
# 'battery_weights', 'stocks', 'shares', 'maintenance_material_fractions', 'lifetimes']



In [38]:
# CREATE XARRAY to store data
all_variables = list(target_variables_narrow.keys()) + list(target_variables_slow.keys())

vehicle_targets = xr.DataArray(
            0.0,
            dims=("year", "region", "variable"),
            coords={"year": [2020,2060],
                    "region": target_regions,
                    "variable": all_variables})

In [49]:
image_output = pd.read_csv("IMAGE scenario explorer variables.csv", delimiter=";", index_col=0, usecols=range(8))
image_output["unit"] = image_output["unit"].astype(str).str.strip() 
image_output["unit"].unique()

array(['Mt/yr', 'Mt', 'km', 'Mt CO2/yr', 'Mt CO2-equiv/yr', 'bn m2',
       'billion tkm/yr', 'billion pkm/yr', 'billion USD_2010/yr',
       'million', 'US$2010/GJ', 'EJ/yr', 'm^2', 'thousand'], dtype=object)

In [ ]:
image_output = pd.read_csv("IMAGE scenario explorer variables.csv", delimiter=";", index_col=0, usecols=range(8))

# Initialize pint's unit registry
ureg = pint.UnitRegistry()

# Sample unit mapping (from image_output DataFrame)
unit_mapping = dict(zip(image_output["variable"], image_output["unit"]))

# Function to check if a unit is valid
def is_valid_unit(unit):
    try:
        ureg(unit)  # Try parsing the unit
        return True
    except pint.errors.UndefinedUnitError:
        return False
    except Exception as e:
        print(f"Unexpected error for unit '{unit}': {e}")
        return False

# Validate each unit
invalid_units = {var: unit for var, unit in unit_mapping.items() if not is_valid_unit(unit)}

# Print invalid units
if invalid_units:
    print("Invalid units found:", invalid_units)
else:
    print("All units are valid!")

    'billion tkm/yr'
    'billion pkm/yr'
    'billion pkm/yr'
    'million'

Invalid units found: {'Emissions|CO2': 'Mt CO2/yr', 'Emissions|Kyoto Gases': 'Mt CO2-equiv/yr', 'Energy Service|Commercial|Floor Space': 'bn m2', 'Energy Service|Residential and Commercial|Floor Space': 'bn m2', 'Energy Service|Residential|Floor Space': 'bn m2', 'Energy Service|Transportation|Freight': 'billion tkm/yr', 'Energy Service|Transportation|Passenger': 'billion pkm/yr', 'GDP|MER': 'billion USD_2010/yr', 'Population': 'million', 'Price|Primary Energy|Biomass': 'US$2010/GJ', 'Price|Primary Energy|Coal': 'US$2010/GJ', 'Price|Primary Energy|Gas': 'US$2010/GJ', 'Price|Primary Energy|Oil': 'US$2010/GJ', 'Product Stock|Transportation|Vehicle Stock|Air|Cargo': 'thousand', 'Product Stock|Transportation|Vehicle Stock|Air|Passenger': 'thousand', 'Product Stock|Transportation|Vehicle Stock|Rail|Cargo': 'thousand', 'Product Stock|Transportation|Vehicle Stock|Rail|Passenger': 'thousand', 'Product Stock|Transportation|Vehicle Stock|Road|Bicycles': 'thousand', 'Product Stock|Transportation|V

In [46]:
# Import model output data
image_output = pd.read_csv("IMAGE scenario explorer variables.csv", delimiter=";", index_col=0, usecols=range(8))

# TRANSFORM model output to xarray

# Define a unit registry
unit_registry = pint_xarray.unit_registry

# Assuming image_output is a Pandas DataFrame with 'variable' and 'unit' columns
unit_mapping = dict(zip(image_output["variable"], image_output["unit"]))

image_output = image_output.drop(columns=["model", "scenario","unit"])

# Step 2: Convert DataFrame into xarray Dataset
xr_image_output = image_output.set_index(["variable", "region", "year"]).to_xarray()

xr_image_output = xr_image_output.sel(year=[2020, 2060])

xr_image_output = xr_image_output.to_dataarray("var2").drop_vars("var2").squeeze()

#xr_image_output = xr_image_output.pint.quantify(unit_mapping)

In [36]:
xr_image_output = xr_image_output.pint.quantify()

In [31]:
# REGIONAL AGGREGATION

# Convert the region names in xr_image_output to match the target regions
xr_regions = xr_image_output.coords["region"].values  # Extract current region names

# Create a mapping of only existing regions
region_mapping_filtered = {key: val for key, val in region_mapping.items() if key in xr_regions}

# Map the dataset’s regions to the target regions
xr_image_output_filtered = xr_image_output.sel(region=list(region_mapping_filtered.keys()))
xr_image_output_filtered = xr_image_output_filtered.assign_coords(region=[region_mapping_filtered[r] for r in xr_image_output_filtered["region"].values])

# Aggregate by summing over the mapped regions
xr_image_output_aggregated = xr_image_output_filtered.groupby("region").sum()

xr_image_output_aggregated.coords["region"] = xr_image_output_aggregated.coords["region"].reindex_like(vehicle_targets.coords["region"] )


In [32]:
xr_image_output_aggregated

<xarray.DataArray (var2: 2, variable: 245, region: 13, year: 2)> Size: 102kB
array([[[['Mt CO2/yr', 'Mt CO2/yr'],
         ['Mt CO2/yr', 'Mt CO2/yr'],
         ['Mt CO2/yr', 'Mt CO2/yr'],
         ...,
         ['Mt CO2/yrMt CO2/yrMt CO2/yr', 'Mt CO2/yrMt CO2/yrMt CO2/yr'],
         ['Mt CO2/yr', 'Mt CO2/yr'],
         ['Mt CO2/yr', 'Mt CO2/yr']],

        [['Mt CO2-equiv/yr', 'Mt CO2-equiv/yr'],
         ['Mt CO2-equiv/yr', 'Mt CO2-equiv/yr'],
         ['Mt CO2-equiv/yr', 'Mt CO2-equiv/yr'],
         ...,
         ['Mt CO2-equiv/yrMt CO2-equiv/yrMt CO2-equiv/yr',
          'Mt CO2-equiv/yrMt CO2-equiv/yrMt CO2-equiv/yr'],
         ['Mt CO2-equiv/yr', 'Mt CO2-equiv/yr'],
         ['Mt CO2-equiv/yr', 'Mt CO2-equiv/yr']],

        [['bn m2', 'bn m2'],
         ['bn m2', 'bn m2'],
         ['bn m2', 'bn m2'],
...
         ...,
         [105.79900169, 741.0605164],
         [89.80000305, 166.7955017],
         [137.154007, 222.8327026]],

        [[0.000436911, 0.000709363],
         [0.000828746, 0.001526832],
         [0.009201959, 0.019278991],
         ...,
         [0.000904502, 0.005950715],
         [0.00323472, 0.005790935],
         [0.003666401, 0.006307198]],

        [[0.00121089, 0.002990272],
         [0.001804333, 0.003638552],
         [0.013587677, 0.040023945],
         ...,
         [0.001506523, 0.012655218],
         [0.014436072, 0.026242287],
         [0.011628404, 0.021815019]]]], dtype=object)
Coordinates:
  * variable  (variable) object 2kB 'Emissions|CO2' ... 'Value Added|Services'
  * year      (year) int64 16B 2020 2060
  * region    (region) object 104B 'Canada' ... 'Subsaharan Africa'
Dimensions without coordinates: var2

In [33]:
pop

<xarray.DataArray (var2: 2, region: 13, year: 2)> Size: 416B
array([[['million', 'million'],
        ['million', 'million'],
        ['million', 'million'],
        ['million', 'million'],
        ['million', 'million'],
        ['millionmillion', 'millionmillion'],
        ['millionmillion', 'millionmillion'],
        ['millionmillion', 'millionmillion'],
        ['millionmillion', 'millionmillion'],
        ['millionmillion', 'millionmillion'],
        ['millionmillionmillion', 'millionmillionmillion'],
        ['million', 'million'],
        ['million', 'million']],

       [[37.88871002, 43.42338181],
        [120.3700027, 101.3702011],
        [1460.223022, 1235.235962],
        [1396.386963, 1630.067017],
        [125.2447968, 99.49688721],
        [1302.8743134, 1498.587799],
        [478.6757812, 719.9852906],
        [961.5216065, 1128.375885],
        [1315.6244391999999, 1340.0663791799998],
        [512.9285202, 501.9851456],
        [1094.5870209, 2425.407043],
        [335.9419861, 387.9917908],
        [422.648407, 434.8951111]]], dtype=object)
Coordinates:
    variable  <U10 40B 'Population'
  * year      (year) int64 16B 2020 2060
  * region    (region) object 104B 'Canada' ... 'Subsaharan Africa'
Dimensions without coordinates: var2

In [34]:
# EXTRACT SERVICE DEMAND
pop = xr_image_output_aggregated.sel(variable="Population")  # Extract Population data
energy_service = xr_image_output_aggregated.sel(variable=[v for v in xr_image_output_aggregated["variable"].values if "Energy Service|Transportation|" in v or "Product Stock|Transportation|" in v])

energy_service_per_capita = energy_service / pop

vehicle_targets.loc[dict(variable="Total freight. travel demand")] = energy_service_per_capita.sel(variable="Energy Service|Transportation|Freight")
vehicle_targets.loc[dict(variable="Total pass. travel demand")] = energy_service_per_capita.sel(variable="Energy Service|Transportation|Passenger")


TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
energy_service

<xarray.DataArray (var2: 2, variable: 16, region: 13, year: 2)> Size: 7kB
array([[[['billion tkm/yr', 'billion tkm/yr'],
         ['billion tkm/yr', 'billion tkm/yr'],
         ['billion tkm/yr', 'billion tkm/yr'],
         ['billion tkm/yr', 'billion tkm/yr'],
         ['billion tkm/yr', 'billion tkm/yr'],
         ['billion tkm/yrbillion tkm/yr',
          'billion tkm/yrbillion tkm/yr'],
         ['billion tkm/yrbillion tkm/yr',
          'billion tkm/yrbillion tkm/yr'],
         ['billion tkm/yrbillion tkm/yr',
          'billion tkm/yrbillion tkm/yr'],
         ['billion tkm/yrbillion tkm/yr',
          'billion tkm/yrbillion tkm/yr'],
         ['billion tkm/yrbillion tkm/yr',
          'billion tkm/yrbillion tkm/yr'],
         ['billion tkm/yrbillion tkm/yrbillion tkm/yr',
          'billion tkm/yrbillion tkm/yrbillion tkm/yr'],
         ['billion tkm/yr', 'billion tkm/yr'],
         ['billion tkm/yr', 'billion tkm/yr']],

...
         [8453.418264, 23217.380669],
         [49717.562369, 79568.50525],
         [4514.614907, 8919.545473],
         [2582.9528454, 13070.007547],
         [18257.24121, 28245.91781],
         [15378.68565, 25070.89432]],

        [[1.242779303, 0.43664049],
         [2.572948617, 0.918893512],
         [8.243221809, 3.340364876],
         [1.302480175, 0.951835522],
         [2.207682448, 0.642924397],
         [5.782295474, 3.117518278],
         [3.106681125, 1.4766322680000001],
         [5.6501164379999995, 3.203117089],
         [33.747580039, 11.786001362],
         [1.900639811, 0.774394356],
         [0.9281843000000001, 1.09322551],
         [5.589143018, 1.910590765],
         [18.49785931, 6.281657543]]]], dtype=object)
Coordinates:
  * variable  (variable) object 128B 'Energy Service|Transportation|Freight' ...
  * year      (year) int64 16B 2020 2060
  * region    (region) object 104B 'Canada' ... 'Subsaharan Africa'
Dimensions without coordinates: var2

In [ ]:
# WEIGHTS CALCULATION
vehicle_weights = vehicle_pre["vehicle_weights"]

# Convert the region names in xr_image_output to match the target regions
xr_type = vehicle_weights.coords["Type"].values  # Extract current region names

# Create a mapping of only existing regions
vehicle_type_mapping_filtered = {key: val for key, val in vehicle_type_mapping.items() if key in xr_type}

# Map the dataset’s regions to the target regions
vehicle_weights_filtered = vehicle_weights.sel(Type=list(vehicle_type_mapping_filtered.keys()))
vehicle_weights_filtered = vehicle_weights_filtered.assign_coords(Type=[vehicle_type_mapping_filtered[r] for r in vehicle_weights_filtered["Type"].values])

# Select only the years 2020 and 2060
vehicle_weights_filtered = vehicle_weights_filtered.sel(Cohort=[2020, 2060])

# Exclude zeros by applying a mask (only non-zero values will be considered in the mean)
vehicle_weights_filtered_no_zeros = vehicle_weights_filtered.where(vehicle_weights_filtered != 0)

# Aggregate by summing over the mapped regions, excluding zeros
vehicle_weights_aggregated = vehicle_weights_filtered_no_zeros.groupby("Type").mean()




In [ ]:
vehicle_targets.name = "values"

# Select the variables you're interested in from vehicle_targets
selected_variables = ["Total pass. travel demand", "Total freight. travel demand"]

# Extract data for the selected variables
selected_data = vehicle_targets.sel(variable=selected_variables)

# Convert to pandas DataFrame
df = selected_data.to_dataframe().reset_index()

# Pivot the DataFrame to have 'year' as columns
df_pivot = df.pivot_table(index=["region", "variable"], columns="year", values="values")


df_pivot.to_csv("travel_demand.csv")
# Display the DataFrame


In [ ]:
# Iterate over the target variables and assign weights to vehicle_targets
for target_var in target_variables_slow:
    if target_var in vehicle_weights_aggregated.coords["Type"].values:
        # Extract the aggregated weight for the corresponding target variable
        weight = vehicle_weights_aggregated.sel(Type=target_var).values
        print(weight)
        
        # Create an array of the same weight for all regions
        broadcasted_weights = np.full(len(vehicle_targets.coords["region"]), weight)
        
        # Assign this array of broadcasted weights to the variable in vehicle_targets
        vehicle_targets.loc[:, target_var, :] = broadcasted_weights


[1500.    1372.526]


ValueError: could not broadcast input array from shape (2,) into shape (13,)